# Source-Aware OSINT Agent with Pydantic AI

The presentation shows a more production-like system: case folders, `AGENTS.md`, an investigation playbook, separate skills, logs, and reports. This notebook keeps the same idea, but compresses it into one Colab-friendly flow:

**case input → source-aware agent → registry/database tools + Pydantic provider-native web search → structured report**


## How this maps to the presentation

| In the presentation | In this notebook |
|---|---|
| `AGENTS.md` | The agent instructions cell |
| `INVESTIGATION_PLAYBOOK.md` | Tool docstrings + workflow rules |
| `skills/*/SKILL.md` | Custom Python tools such as YC World and Aleph lookup |
| Orchestrator skill | The Pydantic AI `Agent` deciding what to call |
| Case folder / `CASE.md` | The company + question prompt |
| Final investigation report | The `ResearchReport` Pydantic output model |

The biggest simplification: we do **not** write a folder-based state machine here. We let Pydantic AI handle provider-native web search and tool calling, then we force the answer into a structured report.


In [ ]:
# Setup cell
%pip install -q requests==2.32.5 "pydantic-ai-slim[openai]==0.8.1" --quiet


In [ ]:
# Paste keys into the empty strings 

import json
import os
import re
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from typing import Literal

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.native_tools import WebSearchTool
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.settings import ModelSettings

OPENAI_API_KEY = ""
YC_WORLD_API_KEY = ""
ALEPH_API_KEY = ""
OPENAI_MODEL = "gpt-4.1-mini"
YC_WORLD_BASE_URL = "https://api.youcontrol.world"
ALEPH_BASE_URL = "https://aleph.occrp.org"
OUTPUT_DIR = Path("outputs")

load_dotenv(Path(".env"))

for key, value in [
    ("OPENAI_API_KEY", OPENAI_API_KEY),
    ("YC_WORLD_API_KEY", YC_WORLD_API_KEY),
    ("ALEPH_API_KEY", ALEPH_API_KEY),
]:
    if value:
        os.environ[key] = value

for key, label in [
    ("OPENAI_API_KEY", "OpenAI API key"),
    ("YC_WORLD_API_KEY", "YC World API key (Enter to skip)"),
    ("ALEPH_API_KEY", "Aleph API key (Enter to skip; public search may still work)"),
]:
    if not os.getenv(key):
        value = getpass(f"{label}: ")
        if value:
            os.environ[key] = value

OPENAI_MODEL = os.getenv("OPENAI_MODEL", OPENAI_MODEL)
YC_WORLD_API_KEY = os.getenv("YC_WORLD_API_KEY", YC_WORLD_API_KEY)
ALEPH_API_KEY = os.getenv("ALEPH_API_KEY", ALEPH_API_KEY) or os.getenv("ALEPHCLIENT_API_KEY", "")
YC_WORLD_BASE_URL = os.getenv("YC_WORLD_BASE_URL", YC_WORLD_BASE_URL).rstrip("/")
ALEPH_BASE_URL = os.getenv("ALEPH_BASE_URL", ALEPH_BASE_URL).rstrip("/")

print(f"Model: OpenAI Responses API / {OPENAI_MODEL}")
print(f"YC World: {'enabled' if YC_WORLD_API_KEY else 'disabled'}")
print(f"Aleph: {'key provided' if ALEPH_API_KEY else 'public/no-key mode'}")


## Report Shape

This cell is like the **report template**.

Pydantic forces the model to separate facts, claims, leads, hypotheses, and discourages one big blurry answer.


In [ ]:

class ResearchReport(BaseModel):
    short_answer: str = Field(description="A cautious direct answer to the research question.")
    registry_records: list[str] = Field(default_factory=list, description="Concrete registry rows with company IDs, dates, roles, source IDs, and raw JSON paths.")
    registry_facts: list[str] = Field(default_factory=list, description="Facts supported by registry or database records.")
    public_claims: list[str] = Field(default_factory=list, description="Claims from public sources such as articles, reports, websites, or PDFs.")
    leads_to_review: list[str] = Field(default_factory=list, description="Promising but unconfirmed leads that need manual review.")
    risk_hypotheses: list[str] = Field(default_factory=list, description="Reasoned hypotheses, clearly not conclusions.")
    uncertainty: list[str] = Field(default_factory=list, description="What remains unclear or weakly supported.")
    next_steps: list[str] = Field(default_factory=list, description="Specific manual checks a reporter should do next.")
    source_ledger: list[str] = Field(default_factory=list, description="URLs, source names, record IDs, database names, and raw JSON paths used.")


## Custom Tools

This cell is like the **skills folder**.

The notebook now has only special-source tools here. General web search is exposed through Pydantic AI's provider-native `WebSearchTool`.


In [ ]:

# Helper used by the registry tools.

CASE_RAW_DIR = None

COUNTRY_ALIASES = {
    "poland": "pl",
    "polska": "pl",
    "pl": "pl",
    "russia": "ru",
    "russian federation": "ru",
    "ru": "ru",
    "ukraine": "ua",
    "ua": "ua",
    "united kingdom": "gb",
    "uk": "gb",
    "gb": "gb",
    "germany": "de",
    "de": "de",
    "france": "fr",
    "fr": "fr",
}

COMPANY_ID_FIELDS = [
    ("registrationNumber", "registration/KRS/OGRN"),
    ("taxNumber", "tax/INN"),
    ("vatCode", "VAT"),
    ("innCode", "INN"),
    ("ogrnCode", "OGRN"),
    ("leiCode", "LEI"),
    ("kppCode", "KPP"),
]


def slug(text: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "_", text.lower()).strip("_") or "company"


def case_folder(company: str) -> Path:
    return OUTPUT_DIR / slug(company)


def set_case_raw_dir(company: str) -> Path:
    global CASE_RAW_DIR
    CASE_RAW_DIR = case_folder(company) / "raw"
    CASE_RAW_DIR.mkdir(parents=True, exist_ok=True)
    return CASE_RAW_DIR


def raw_output_dir() -> Path:
    folder = CASE_RAW_DIR or (OUTPUT_DIR / "_uncategorized" / "raw")
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def unique_json_path(stem: str) -> Path:
    folder = raw_output_dir()
    safe_stem = slug(stem)
    path = folder / f"{safe_stem}.json"
    counter = 2
    while path.exists():
        path = folder / f"{safe_stem}_{counter}.json"
        counter += 1
    return path


def save_raw_json(source: str, stem: str, request: dict, response_json: dict) -> Path:
    path = unique_json_path(stem)
    payload = {
        "saved_at": datetime.now(timezone.utc).isoformat(),
        "source": source,
        "request": request,
        "response": response_json,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def normalize_country(country: str = "") -> str:
    value = (country or "").strip().lower()
    return COUNTRY_ALIASES.get(value, value)


def comparable_name(text: str) -> str:
    return re.sub(r"[^0-9a-z]+", "", (text or "").lower())


def entity_rank(query: str, entity: dict) -> tuple[int, str]:
    query_key = comparable_name(query)
    names = entity_names(entity, limit=8).split(" / ")
    keys = [comparable_name(name) for name in names]
    if query_key in keys:
        return (0, entity_names(entity))
    if any(key.startswith(query_key) or query_key.startswith(key) for key in keys if key):
        return (1, entity_names(entity))
    return (2, entity_names(entity))


def values(value, limit: int = 4) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value[:limit] if item not in (None, "")]
    return [str(value)] if value not in (None, "") else []


def first(value) -> str:
    vals = values(value, limit=1)
    return vals[0] if vals else ""


def joined(value, limit: int = 4) -> str:
    return ", ".join(values(value, limit=limit))


def item_schema(entity: dict) -> str:
    item = (entity.get("items") or [{}])[0]
    return item.get("schema") or entity.get("schema") or "Entity"


def is_person_schema(schema: str) -> bool:
    return schema.lower() in {"person", "human"}


def entity_names(entity: dict, limit: int = 4) -> str:
    names = []
    for item in entity.get("items", []):
        props = item.get("properties", {}) or {}
        for name in values(props.get("name"), limit=2):
            if name and name not in names:
                names.append(name)
        caption = item.get("caption")
        if caption and caption not in names:
            names.append(caption)
    caption = entity.get("caption")
    if caption and caption not in names:
        names.append(caption)
    return " / ".join(names[:limit])


def company_id_bits(props: dict, schema: str) -> list[str]:
    if is_person_schema(schema):
        return []
    bits = []
    for key, label in COMPANY_ID_FIELDS:
        value = joined(props.get(key), limit=3)
        if value:
            bits.append(f"{label}: {value}")
    return bits


def entity_summary(prefix: str, entity: dict, external_id: str = "") -> str:
    item = (entity.get("items") or [{}])[0]
    props = item.get("properties", {}) or {}
    schema = item_schema(entity)
    name = entity_names(entity) or first(props.get("name")) or "unknown"

    bits = [f"{prefix}: {name}", f"schema: {schema}"]
    bits.extend(company_id_bits(props, schema))

    for key, label in [
        ("country", "country"),
        ("jurisdiction", "jurisdiction"),
        ("legalForm", "legal form"),
        ("incorporationDate", "incorporated"),
        ("status", "status"),
        ("address", "address"),
        ("sourceUrl", "source URL"),
        ("publisher", "publisher"),
    ]:
        value = joined(props.get(key), limit=2 if key == "address" else 4)
        if value:
            bits.append(f"{label}: {value}")

    if external_id:
        bits.append(f"YC World ID: {external_id}")
    return "; ".join(bits)


def relation_summaries(relation_entity: dict, max_items: int = 3) -> list[str]:
    relation_data = relation_entity.get("relationData", {}) or {}
    relation_schema = relation_data.get("schema", "related")
    lines = []
    for item in (relation_data.get("items") or [])[:max_items]:
        props = item.get("properties", {}) or {}
        bits = [f"relation schema: {relation_schema or item.get('schema', 'related')}"]
        for key, label in [
            ("role", "role"),
            ("percentage", "ownership"),
            ("sharesValue", "shares value"),
            ("startDate", "start date"),
            ("endDate", "end date"),
            ("sourceUrl", "source URL"),
            ("publisher", "publisher"),
            ("description", "description"),
            ("summary", "summary"),
        ]:
            value = joined(props.get(key), limit=4)
            if value:
                bits.append(f"{label}: {value}")
        lines.append("  Relation detail: " + "; ".join(bits))
    return lines


def yc_world_entity_detail(external_id: str) -> str:
    """
    Fetch one full YC World entity record using /Entity/{externalId}/get-entity.

    Use this after yc_world_lookup when you need directors, founders, owners,
    authorized signatories, related companies, relation roles, or relation dates.
    The returned text summarizes relationsData; the full JSON is saved to disk.
    """
    if not YC_WORLD_API_KEY:
        return "YC World is disabled because YC_WORLD_API_KEY is not set."
    if not external_id:
        return "YC World entity detail needs an external_id from yc_world_lookup."

    request_metadata = {"url": f"{YC_WORLD_BASE_URL}/Entity/{external_id}/get-entity"}
    response = requests.get(
        request_metadata["url"],
        headers={"Accept": "application/json", "x-api-key": YC_WORLD_API_KEY},
        timeout=45,
    )
    if not response.ok:
        return "Full entity detail: unavailable"

    raw_response = response.json()
    raw_path = save_raw_json("YC World", f"yc_world_entity_detail_{external_id}", request_metadata, raw_response)
    data = raw_response.get("result", raw_response)

    lines = [f"Full Entity Raw JSON: {raw_path}"]
    lines.append(entity_summary("Main entity", {"items": data.get("items", [])}, external_id))

    counts = data.get("relationsCount") or []
    if counts:
        count_text = ", ".join(f"{row.get('schema', 'unknown')}={row.get('count', '?')}" for row in counts)
        lines.append(f"Relation counts: {count_text}")

    for relation in (data.get("relationsData") or [])[:12]:
        lines.append(entity_summary("Related entity", relation))
        lines.extend(relation_summaries(relation))

    return "\n".join(lines)


def yc_world_lookup(query: str, schema: Literal["Company", "Person"] = "Company", country: str = "") -> str:
    """
    Search YC World registry records, then summarize full entity detail for each result.
    Use schema='Company' or schema='Person'. Country can be 'PL' or 'Poland'.

    Use for company registry facts, founders, directors, owners, addresses,
    registration numbers, related entities, relation roles, and relation dates.

    Warning: registry matches are leads until manually reviewed. Include source URLs,
    YC World IDs, raw JSON paths, and source identifiers in the report ledger.
    """
    if not YC_WORLD_API_KEY:
        return "YC World is disabled because YC_WORLD_API_KEY is not set."

    normalized_country = normalize_country(country)
    params = {"SearchString": query, "SchemaName": schema, "PageSize": 5, "Offset": 0}
    if normalized_country:
        params["Countries"] = normalized_country
    request_metadata = {"url": f"{YC_WORLD_BASE_URL}/GetEntities", "params": params}

    response = requests.get(
        request_metadata["url"],
        params=params,
        headers={"Accept": "application/json", "x-api-key": YC_WORLD_API_KEY},
        timeout=45,
    )
    response.raise_for_status()
    raw_response = response.json()
    raw_path = save_raw_json("YC World", f"yc_world_{schema.lower()}_search", request_metadata, raw_response)
    payload = raw_response.get("result", raw_response)

    lines = [
        f"Raw JSON: {raw_path}",
        f"YC World search: {query}; schema: {schema}; country: {normalized_country or 'any'}; total results: {payload.get('total', 0)}",
    ]
    entities = sorted(payload.get("entities", []), key=lambda entity: entity_rank(query, entity))
    for entity in entities[:5]:
        external_id = entity.get("externalId", "")
        lines.append(entity_summary("Search result", entity, external_id))
        if external_id:
            lines.append(yc_world_entity_detail(external_id))
        lines.append("")

    if len(lines) == 2:
        lines.append("No YC World results found.")
    return "\n".join(lines)


In [ ]:
# Aleph = another special-source skill.
def aleph_lookup(query: str, schema: Literal["Company", "Person"] = "Company") -> str:
    """
    Search Aleph investigative datasets. Use schema='Company' or schema='Person'.

    Use for investigative datasets, sanctions-style records, leaks,
    PEP-style links, cross-border leads, and related entities.

    Do not use for basic public background that should come from public web sources.

    Warning: Aleph hits are leads unless the underlying dataset is inspected.
    Include Aleph URLs, entity IDs, and dataset names in the report ledger.
    """
    headers = {"Accept": "application/json"}
    if ALEPH_API_KEY:
        headers["Authorization"] = f"ApiKey {ALEPH_API_KEY}"

    params = {"q": query, "filter:schema": schema, "limit": 5}
    request_metadata = {"url": f"{ALEPH_BASE_URL}/api/2/entities", "params": params}
    response = requests.get(
        request_metadata["url"],
        params=params,
        headers=headers,
        timeout=45,
    )
    response.raise_for_status()
    raw_response = response.json()
    raw_path = save_raw_json("Aleph", f"aleph_{query}_{schema}", request_metadata, raw_response)
    rows = raw_response.get("results", [])[:5]

    lines = [f"Raw JSON: {raw_path}"]
    for item in rows:
        links = item.get("links", {}) or {}
        lines.append(f"Entity: {item.get('caption', '?')} ({item.get('schema', '')})")
        lines.append(f"  URL: {links.get('ui', '')}")
        lines.append(f"  Aleph ID: {item.get('id', '')}")
        lines.append("")

    if len(lines) == 1:
        lines.append("No Aleph results found.")
    return "\n".join(lines)


## The Agent

This cell is like `AGENTS.md` plus a simple orchestrator.


In [ ]:
# Concise instructions usually work better than long narrative prompts.
# Tool docstrings carry source-specific guidance.

AGENT_INSTRUCTIONS = """
You are a cautious OSINT research assistant.

Goal:
Answer the user's company-research question using source-led evidence.

Workflow:
1. Start with the provided YC World registry preflight; it includes search results and full entity detail from /get-entity.
2. Do not skip registry evidence when answering ownership or control questions.
3. Use YC World relation details and relation counts for founders, directors, owners, authorized signatories, employment rows, and related entities when available. Do not omit authorized signatories just because they are not shareholders.
4. If yc_world_lookup returns a promising YC World ID, use yc_world_entity_detail for deeper relation rows when needed.
5. Use the custom tools according to their docstrings.
6. Use the provider-native WebSearchTool for public reports, media, company websites, PDFs,
   NGO reports, and official pages.
7. Do not treat search snippets alone as confirmed evidence; prefer sources with URLs or readable source context.
8. Use at least two source types when possible.
9. Follow only the strongest related-person or related-company leads.

Evidence rules:
- Treat matches as leads unless directly confirmed by a source.
- Separate facts, claims, leads, hypotheses, uncertainty, and next steps.
- Include URLs, source IDs, record IDs, database names, and raw JSON filenames or paths in the source ledger.
- Say clearly when evidence is weak, missing, contradictory, or ambiguous.
"""

agent = Agent(
    OpenAIResponsesModel(OPENAI_MODEL),
    output_type=ResearchReport,
    # Provider-native web search, not local DuckDuckGo.
    builtin_tools=[WebSearchTool()],
    tools=[yc_world_lookup, yc_world_entity_detail, aleph_lookup],
    instructions=AGENT_INSTRUCTIONS,
    model_settings=ModelSettings(temperature=0),
)


## Run It

This cell is like the **case file**.

Keep the run prompt short. The agent already has the workflow and evidence rules above, so the prompt only needs the target and the question.


In [ ]:

def save_report(company: str, question: str, report: ResearchReport) -> Path:
    folder = case_folder(company)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "report.md"

    sections = [
        ("Registry Records", report.registry_records),
        ("Registry Facts", report.registry_facts),
        ("Public Claims", report.public_claims),
        ("Leads To Review", report.leads_to_review),
        ("Risk Hypotheses", report.risk_hypotheses),
        ("Uncertainty", report.uncertainty),
        ("Next Steps", report.next_steps),
        ("Source Ledger", report.source_ledger),
    ]

    lines = [
        f"# OSINT report: {company}",
        "",
        f"Question: {question}",
        "",
        "## Short Answer",
        "",
        report.short_answer,
        "",
    ]

    for title, items in sections:
        if items:
            lines.extend([f"## {title}", "", *[f"- {item}" for item in items], ""])

    path.write_text("\n".join(lines), encoding="utf-8")
    return path


async def run_investigation(company: str, question: str, country: str = "") -> ResearchReport:
    raw_dir = set_case_raw_dir(company)
    normalized_country = normalize_country(country)
    yc_world_preflight = yc_world_lookup(company, schema="Company", country=normalized_country)

    prompt = f"""
Target company: {company}
Country: {normalized_country or country or "unknown"}
Research question: {question}

Required YC World registry preflight:
{yc_world_preflight}

Use the YC World preflight when answering ownership or control questions. Put concrete
company IDs, incorporation dates, relation roles, relation dates, source URLs, YC World IDs,
and raw JSON paths in registry_records or source_ledger. Include directorship, ownership,
employment, and authorized-signatory relation rows for the exact target company. Do not include personal national
ID numbers in the final report.

If the YC World preflight is disabled, empty, or inconclusive, explicitly report that registry uncertainty.

Produce a cautious source-led report.
"""
    result = await agent.run(prompt)
    report = result.output
    print(report.short_answer)
    print(f"\nRaw JSON folder: {raw_dir}")
    print(f"\nSaved: {save_report(company, question, report)}")
    return report


In [ ]:

report = await run_investigation(
    company="Consteel Electronics sp. z o.o.",
    country="PL",
    question="Who owns or controls this company, and are there any risk-relevant public claims or leads?"
)
